In [ ]:
from fastai.vision.all import *
path = untar_data(URLs.MNIST_SAMPLE)

<div><progress max="3214948" value="3219456"></progress> 100.14% [3219456/3214948 00:00&lt;00:00]</div>

In [ ]:
# 1. Data Preparation

In [29]:
path.ls()

[Path('/root/.fastai/data/mnist_sample/valid'), Path('/root/.fastai/data/mnist_sample/labels.csv'), Path('/root/.fastai/data/mnist_sample/train')]

In [53]:
print((path/"valid").ls(),(path/"train").ls())


[Path('/root/.fastai/data/mnist_sample/valid/3'), Path('/root/.fastai/data/mnist_sample/valid/7')] [Path('/root/.fastai/data/mnist_sample/train/3'), Path('/root/.fastai/data/mnist_sample/train/7')]


In [60]:
x_train = torch.cat([
    torch.stack( [tensor(Image.open(o)) for o in (path/'train'/'3').ls().sorted()]),
    torch.stack( [tensor(Image.open(o)) for o in (path/'train'/'7').ls().sorted()])
]).float()/255
x_train = x_train.view(-1, 28*28)

x_val = torch.cat ([
    torch.stack( [tensor(Image.open(o)) for o in (path/'valid'/'3').ls().sorted()]),
    torch.stack( [tensor(Image.open(o)) for o in (path/'valid'/'7').ls().sorted()])
]).float()/255
x_val = x_val.view(-1,28*28)

In [51]:
y_train = tensor([1]*len((path/'train'/'3').ls())+[0]*len( (path/'train'/'7').ls())).float()
y_val = tensor( [1]*len((path/'valid'/'3').ls())+[0]*len((path/'valid'/'7').ls())).float()

In [61]:
train_dset = list(zip(x_train,y_train))
val_dset = list(zip(x_val, y_val))

In [67]:
dls = DataLoaders(
    DataLoader(train_dset, batch_size=4, shuffle=True),
    DataLoader(val_dset, batch_size=4, shuffle= False)
)

In [ ]:
# 2. Loss and Metrics

In [63]:
def mnist_loss(predictions, targets):
    predictions = predictions.sigmoid()
    return torch.where(targets==1, 1-predictions, predictions).mean()

In [58]:
def batch_accuracy(xb, yb):
    preds = xb.sigmoid()
    correct = (preds>0.5) == yb
    return correct.float().mean()

In [ ]:
# 3. Train

In [69]:
learn = Learner(dls, nn.Linear(28*28, 1), opt_func=SGD,
                loss_func=mnist_loss, metrics=batch_accuracy)
learn.fit(5, lr=0.1)

epoch,train_loss,valid_loss,batch_accuracy,time
0,0.384867,0.038246,0.973013,00:05
1,0.360210,0.031182,0.977429,00:06
2,0.374571,0.029430,0.976448,00:05
3,0.380994,0.027748,0.978410,00:05
4,0.405437,0.027448,0.977429,00:05
